In [13]:
# Local environment check for QuimicaAutomocio P2.
# The original eChem notebooks are Colab-oriented; this one is designed for local use.
import importlib
import numpy as np
import py3Dmol
import veloxchem as vlx

required = ["XtbDriver", "OptimizationDriver", "VibrationalAnalysis"]
missing = [name for name in required if not hasattr(vlx, name)]
if missing:
    raise RuntimeError(f"VeloxChem is missing required classes: {missing}")

# If xtb-python was installed after the kernel started, restart the kernel.
import xtb  # noqa: F401

print("VeloxChem", getattr(vlx, "__version__", "unknown"))
print("Environment OK")

VeloxChem 1.0rc4
Environment OK


In [14]:
molecule = vlx.Molecule.read_smiles("CCCC(CCO)c1c(C)[nH]c2ccccc12")

print(molecule.get_xyz_string())
molecule.show(atom_indices=True, width=520, height=360)

38

C              1.007580000000         1.539867000000        -3.060736000000
C              0.484673000000         1.991588000000        -1.697041000000
C             -0.819491000000         1.277428000000        -1.296095000000
C             -0.608246000000        -0.182032000000        -0.819413000000
C             -1.954059000000        -0.935434000000        -0.649098000000
C             -2.406100000000        -1.663023000000        -1.925071000000
O             -2.682224000000        -0.759427000000        -2.959812000000
C              0.189742000000        -0.219948000000         0.472324000000
C              1.480485000000        -0.733751000000         0.644463000000
C              2.330667000000        -1.351999000000        -0.420703000000
N              1.892992000000        -0.583353000000         1.923854000000
C              0.879899000000         0.014258000000         2.571022000000
C              0.841749000000         0.371262000000         3.918225000000
C       

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [15]:
xtb_drv = vlx.XtbDriver()
xtb_drv.ostream.mute()

opt_drv = vlx.OptimizationDriver(xtb_drv)
opt_drv.ostream.mute()
opt_drv.max_iter = 100

opt_results = opt_drv.compute(molecule)
opt_molecule = opt_results["final_molecule"]

print("Optimized geometry:")
print(opt_molecule.get_xyz_string())
print("Optimization steps:", len(opt_results.get("opt_energies", [])))
opt_molecule.show(atom_indices=True, width=520, height=360)

Optimized geometry:
38

C              1.142929614499         1.521035946930        -2.863535435850
C              0.359981293887         1.976420497474        -1.637602235618
C             -0.888391147329         1.134106968961        -1.375203508545
C             -0.569927787208        -0.287279829090        -0.889012992360
C             -1.841983571859        -1.131398178732        -0.698509314822
C             -2.537599836134        -1.519677978155        -2.005488063770
O             -3.210779747322        -0.460181966495        -2.644543532269
C              0.222995181311        -0.260951063011         0.384671055409
C              1.489781104620        -0.751437657020         0.560146736604
C              2.421531027849        -1.414145910972        -0.393879580824
N              1.889294785083        -0.539970084981         1.858357857341
C              0.890520331468         0.084170857495         2.545339084146
C              0.838353290799         0.491438010545         3.8

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [16]:
vib_drv = vlx.VibrationalAnalysis(xtb_drv)
vib_drv.ostream.mute()
vib_drv.do_ir = True

vib_results = vib_drv.compute(opt_molecule)

frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
normal_modes = np.array(vib_results["normal_modes"], dtype=float)
reduced_masses = np.array(vib_results["reduced_masses"], dtype=float)
force_constants = np.array(vib_results["force_constants"], dtype=float)
ir_intensities = np.array(vib_results.get("ir_intensities", np.zeros_like(frequencies)), dtype=float)

print("mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1")
for i, freq in enumerate(frequencies, start=1):
    print(f"{i:>4d}  {freq:>16.2f}  {reduced_masses[i-1]:>15.4f}  {force_constants[i-1]:>25.4f}  {ir_intensities[i-1]:>14.2f}")

mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1
   1             22.80           2.9578                     0.0009            0.11
   2             35.00           4.2788                     0.0031            2.34
   3             53.58           2.2579                     0.0038            1.14
   4             63.91           2.3100                     0.0056            0.29
   5             74.70           1.6760                     0.0055            0.26
   6             99.84           2.1159                     0.0124            2.05
   7            122.52           3.2001                     0.0283            0.23
   8            126.13           2.6711                     0.0250            0.79
   9            153.56           2.8907                     0.0402            0.20
  10            171.62           2.8586                     0.0496            0.68
  11            189.36           3.6016                     0.0761            1.32
 

In [21]:
def normal_mode_trajectory(molecule, mode, amplitude=0.45, frames=32):
    labels = list(molecule.get_labels())
    coords = np.array(molecule.get_coordinates_in_angstrom(), dtype=float)
    mode = np.array(mode, dtype=float)

    chunks = []
    for phase in np.linspace(0.0, 2.0 * np.pi, frames, endpoint=False):
        displaced = coords + amplitude * np.sin(phase) * mode
        chunks.append(str(len(labels)))
        chunks.append("normal-mode frame")
        for label, xyz in zip(labels, displaced):
            chunks.append(f"{label:2s} {xyz[0]: .8f} {xyz[1]: .8f} {xyz[2]: .8f}")
    return "\n".join(chunks) + "\n"

def show_mode(molecule, mode_index, amplitude=0.45, frames=32):
    traj = normal_mode_trajectory(molecule, normal_modes[mode_index], amplitude=amplitude, frames=frames)
    view = py3Dmol.view(width=650, height=430)
    view.addModelsAsFrames(traj, "xyz")
    view.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.24}})
    view.animate({"loop": "backAndForth", "reps": 0, "interval": 80})
    view.zoomTo()
    return view

# Python index: 0 is the first vibrational mode. Change this value.
mode_index = 106
print(f"Showing mode {mode_index}: {frequencies[mode_index]:.2f} cm^-1")
show_mode(opt_molecule, mode_index, amplitude=0.45, frames=32).show()
print(normal_modes[mode_index])

Showing mode 106: 3482.36 cm^-1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

[[-2.86086268e-06  6.54650876e-06  6.18845610e-06]
 [-1.82185449e-05 -1.68619248e-06 -1.87106071e-05]
 [ 7.55205815e-06  1.49772223e-07 -4.40159518e-06]
 [ 3.44211981e-05 -1.00203934e-05 -2.36645132e-05]
 [ 1.82081848e-06  2.50531957e-06 -1.42468166e-05]
 [-3.45466782e-06  3.10498902e-06  3.03889197e-06]
 [-6.14387108e-06  3.40939254e-06  4.30141157e-06]
 [-9.89863131e-04  4.49802789e-04  1.53135041e-04]
 [ 1.89946736e-03 -1.06488442e-03 -8.40348524e-04]
 [ 1.94104868e-05  7.20624050e-05  3.95370687e-05]
 [ 6.28648693e-02 -1.87481062e-02  2.66809763e-02]
 [ 7.80054669e-04  1.77761697e-04  1.73499037e-03]
 [-2.48941144e-04  1.97994488e-04  3.28338853e-04]
 [ 3.46652782e-04 -1.38601249e-04  2.90660411e-05]
 [-1.92669130e-04  8.49968097e-06 -2.46683786e-04]
 [-7.49614053e-05  1.21139470e-04  3.14465689e-04]
 [-2.81876542e-04 -1.25127082e-04 -8.74142766e-04]
 [-7.01487297e-05  5.46216440e-06 -9.12211263e-05]
 [ 3.26592187e-05  8.21121776e-07  6.01206755e-06]
 [ 9.12882991e-05 -9.64357757e-